# Used Car Price Predictor (EGP)

End-to-end project: data generation, EDA, training, evaluation, testing, and an interactive predictor.

**Note:** the dataset is synthetic (generated by a realistic depreciation formula plus noise). To use real data, set `USE_REAL_DATA = True` in the data cell and upload a CSV with the columns `age`, `hp`, `km`, `price`.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

SEED = 42
np.random.seed(SEED)
pd.options.display.float_format = "{:,.2f}".format

## 2. Data

In [ ]:
USE_REAL_DATA = False  # set to True to upload your own CSV (columns: age, hp, km, price)

def generate_synthetic_data(n=6000, seed=SEED):
    rng = np.random.default_rng(seed)
    age = rng.integers(0, 16, n)
    hp = np.clip(rng.normal(170, 50, n), 80, 350).round().astype(int)
    # Mileage grows with age (about 15k km per year, with variation)
    km = np.clip(age * rng.normal(15000, 5000, n) + rng.normal(0, 4000, n), 0, 200000)
    km = (km // 1000 * 1000).astype(int)

    base_price = 350000 + hp * 4500                 # engine power sets the base price
    age_factor = 0.90 ** age                        # about 10% depreciation per year
    km_factor = 1 - (km / 200000) * 0.30            # up to 30% loss at 200k km
    noise = rng.normal(1.0, 0.06, n)                # market noise
    price = base_price * age_factor * km_factor * noise
    price = np.round(price / 1000) * 1000

    return pd.DataFrame({"age": age, "hp": hp, "km": km, "price": price})

if USE_REAL_DATA:
    from google.colab import files
    uploaded = files.upload()
    df = pd.read_csv(next(iter(uploaded)))
else:
    df = generate_synthetic_data()

FEATURES = ["age", "hp", "km"]
TARGET = "price"

missing = [c for c in FEATURES + [TARGET] if c not in df.columns]
assert not missing, f"Missing columns: {missing}"

df.head()

## 3. Data quality checks and exploration

In [ ]:
print("Shape:", df.shape)
print("\nMissing values per column:")
print(df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())

# Basic cleaning
df = df.dropna(subset=FEATURES + [TARGET]).drop_duplicates().reset_index(drop=True)
df = df[(df["price"] > 0) & (df["km"] >= 0) & (df["age"] >= 0)].reset_index(drop=True)
print("\nShape after cleaning:", df.shape)

df.describe()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
axes[0].hist(df["price"], bins=40, color="#4c72b0"); axes[0].set_title("Price distribution")
axes[1].scatter(df["age"], df["price"], s=4, alpha=0.3); axes[1].set_title("Price vs age")
axes[2].scatter(df["km"], df["price"], s=4, alpha=0.3); axes[2].set_title("Price vs km")
axes[3].scatter(df["hp"], df["price"], s=4, alpha=0.3); axes[3].set_title("Price vs horsepower")
plt.tight_layout()
plt.show()

print("Correlation with price:")
print(df[FEATURES + [TARGET]].corr()[TARGET].drop(TARGET))

## 4. Train / test split

In [ ]:
X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print("Train:", X_train.shape, " Test:", X_test.shape)

## 5. Model comparison (5-fold cross-validation on the training set)

The test set stays untouched until the final evaluation.

In [ ]:
# Monotonic constraints: price must not increase with age or km, and must not decrease with hp
# Order follows FEATURES: age (-1), hp (+1), km (-1)
candidates = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=300, min_samples_leaf=3, n_jobs=-1, random_state=SEED),
    "Hist Gradient Boosting": HistGradientBoostingRegressor(
        max_iter=400, learning_rate=0.05, max_leaf_nodes=15,
        monotonic_cst=[-1, 1, -1], random_state=SEED),
}

kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
cv_results = {}
for name, mdl in candidates.items():
    rmse = -cross_val_score(mdl, X_train, y_train, cv=kf, scoring="neg_root_mean_squared_error")
    r2 = cross_val_score(mdl, X_train, y_train, cv=kf, scoring="r2")
    cv_results[name] = {"CV RMSE": rmse.mean(), "CV RMSE std": rmse.std(), "CV R2": r2.mean()}

cv_df = pd.DataFrame(cv_results).T.sort_values("CV RMSE")
cv_df

In [ ]:
best_name = cv_df.index[0]
print("Best model by CV RMSE:", best_name)

model = candidates[best_name]
model.fit(X_train, y_train)

## 6. Final evaluation on the held-out test set

In [ ]:
def evaluate(model, X_eval, y_eval):
    pred = model.predict(X_eval)
    return {
        "MAE (EGP)": mean_absolute_error(y_eval, pred),
        "RMSE (EGP)": np.sqrt(mean_squared_error(y_eval, pred)),
        "R2": r2_score(y_eval, pred),
        "MAPE (%)": np.mean(np.abs((y_eval - pred) / y_eval)) * 100,
    }

train_metrics = evaluate(model, X_train, y_train)
test_metrics = evaluate(model, X_test, y_test)
pd.DataFrame({"Train": train_metrics, "Test": test_metrics})

In [ ]:
pred_test = model.predict(X_test)
residuals = y_test - pred_test

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(y_test, pred_test, s=6, alpha=0.4)
lims = [min(y_test.min(), pred_test.min()), max(y_test.max(), pred_test.max())]
axes[0].plot(lims, lims, "r--")
axes[0].set_xlabel("Actual price"); axes[0].set_ylabel("Predicted price"); axes[0].set_title("Actual vs predicted")

axes[1].hist(residuals, bins=40, color="#55a868")
axes[1].set_title("Residuals (actual - predicted)")

axes[2].scatter(pred_test, residuals, s=6, alpha=0.4)
axes[2].axhline(0, color="r", linestyle="--")
axes[2].set_xlabel("Predicted price"); axes[2].set_ylabel("Residual"); axes[2].set_title("Residuals vs predicted")

plt.tight_layout()
plt.show()

In [ ]:
imp = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=SEED, scoring="r2")
imp_df = pd.Series(imp.importances_mean, index=FEATURES).sort_values()
imp_df.plot(kind="barh", title="Permutation importance (drop in R2)", color="#c44e52", figsize=(6, 3))
plt.show()

## 7. Save and reload the model

In [ ]:
MODEL_PATH = "car_price_model.joblib"
joblib.dump({"model": model, "features": FEATURES}, MODEL_PATH)

bundle = joblib.load(MODEL_PATH)
model = bundle["model"]
FEATURES = bundle["features"]
print("Model saved and reloaded:", MODEL_PATH)

## 8. Prediction helper

In [ ]:
def predict_price(age, hp, km):
    row = pd.DataFrame([[age, hp, km]], columns=FEATURES)
    return float(max(model.predict(row)[0], 0))

print(f"3-year-old car, 180 hp, 40,000 km -> {predict_price(3, 180, 40000):,.0f} EGP")

## 9. Automated tests

Includes a performance threshold, sanity checks on model behavior, and input edge cases.

In [ ]:
results = []

def run_test(name, condition, detail=""):
    results.append({"Test": name, "Result": "PASS" if condition else "FAIL", "Detail": detail})

# Test 1: performance on unseen data
test_r2 = test_metrics["R2"]
run_test("Test R2 >= 0.90", test_r2 >= 0.90, f"R2 = {test_r2:.4f}")

# Test 2: no severe overfitting
gap = train_metrics["R2"] - test_metrics["R2"]
run_test("Train/test R2 gap <= 0.05", gap <= 0.05, f"gap = {gap:.4f}")

# Test 3: older cars cost less (all else equal)
run_test("Price drops with age",
         predict_price(2, 180, 40000) > predict_price(8, 180, 40000),
         f"{predict_price(2, 180, 40000):,.0f} > {predict_price(8, 180, 40000):,.0f}")

# Test 4: more km means lower price
run_test("Price drops with mileage",
         predict_price(4, 180, 20000) > predict_price(4, 180, 150000),
         f"{predict_price(4, 180, 20000):,.0f} > {predict_price(4, 180, 150000):,.0f}")

# Test 5: stronger engine means higher price
run_test("Price rises with horsepower",
         predict_price(4, 250, 60000) > predict_price(4, 110, 60000),
         f"{predict_price(4, 250, 60000):,.0f} > {predict_price(4, 110, 60000):,.0f}")

# Test 6: predictions are always positive over the full slider range
grid = pd.DataFrame(
    [(a, h, k) for a in range(0, 16, 3) for h in range(80, 351, 90) for k in range(0, 200001, 50000)],
    columns=FEATURES)
run_test("All predictions are positive", bool((model.predict(grid) > 0).all()),
         f"min prediction = {model.predict(grid).min():,.0f}")

# Test 7: no NaN output
run_test("No NaN predictions", not np.isnan(model.predict(grid)).any())

# Test 8: save/load gives identical predictions
reloaded = joblib.load(MODEL_PATH)["model"]
run_test("Reloaded model matches original",
         np.allclose(reloaded.predict(X_test), model.predict(X_test)))

# Test 9: median relative error on the test set
median_ape = np.median(np.abs((y_test - pred_test) / y_test)) * 100
run_test("Median error <= 8%", median_ape <= 8, f"median APE = {median_ape:.2f}%")

report = pd.DataFrame(results)
print(f"{(report['Result'] == 'PASS').sum()} / {len(report)} tests passed")
report

## 10. Interactive predictor

In [ ]:
import ipywidgets as widgets
from IPython.display import display

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

style = {"description_width": "initial"}
layout = widgets.Layout(width="500px")

age_slider = widgets.IntSlider(value=3, min=0, max=15, description="Car age (years):", style=style, layout=layout)
hp_slider = widgets.IntSlider(value=180, min=80, max=350, description="Engine power (hp):", style=style, layout=layout)
km_slider = widgets.IntSlider(value=40000, min=0, max=200000, step=5000, description="Mileage (km):", style=style, layout=layout)

def show_prediction(age, hp, km):
    price = predict_price(age, hp, km)
    print(f"Estimated price: {price:,.0f} EGP")
    print(f"Typical range (+/- MAE): {max(price - test_metrics['MAE (EGP)'], 0):,.0f} to {price + test_metrics['MAE (EGP)']:,.0f} EGP")

widgets.interactive(show_prediction, age=age_slider, hp=hp_slider, km=km_slider)

## 11. Test the model with your own numbers

In [ ]:
# Typed-input form: enter values, click the button, get the result
train_ranges = {c: (int(df[c].min()), int(df[c].max())) for c in FEATURES}

age_box = widgets.IntText(value=3, description="Car age (years):", style=style, layout=layout)
hp_box = widgets.IntText(value=180, description="Engine power (hp):", style=style, layout=layout)
km_box = widgets.IntText(value=40000, description="Mileage (km):", style=style, layout=layout)
predict_btn = widgets.Button(description="Predict price", button_style="success", icon="calculator")
out = widgets.Output()

def on_predict(_):
    with out:
        out.clear_output()
        values = {"age": age_box.value, "hp": hp_box.value, "km": km_box.value}

        # Input validation
        errors = [f"{k} must be 0 or more" for k, v in values.items() if v < 0]
        if values["hp"] <= 0:
            errors.append("hp must be greater than 0")
        if errors:
            print("Invalid input:", "; ".join(errors))
            return

        # Warn when inputs fall outside the range the model was trained on
        for k, v in values.items():
            lo, hi = train_ranges[k]
            if not lo <= v <= hi:
                print(f"Warning: {k}={v:,} is outside the training range ({lo:,} to {hi:,}). The result may be unreliable.")

        price = predict_price(**values)
        mae = test_metrics["MAE (EGP)"]
        print("-" * 45)
        print(f"Car age      : {values['age']} years")
        print(f"Engine power : {values['hp']} hp")
        print(f"Mileage      : {values['km']:,} km")
        print("-" * 45)
        print(f"Estimated price : {price:,.0f} EGP")
        print(f"Likely range    : {max(price - mae, 0):,.0f} to {price + mae:,.0f} EGP")

predict_btn.on_click(on_predict)
display(widgets.VBox([age_box, hp_box, km_box, predict_btn, out]))

### Optional: compare several cars at once

Edit the table below with your own cars and run the cell.

In [ ]:
my_cars = pd.DataFrame({
    "age": [1, 5, 10],
    "hp":  [150, 200, 120],
    "km":  [15000, 80000, 190000],
})

my_cars["predicted_price_EGP"] = model.predict(my_cars[FEATURES]).round(-3)
my_cars

### Optional: console-style input

Run this cell and type the numbers when prompted. Leave the first answer empty to stop.

In [ ]:
while True:
    raw = input("Car age in years (empty to stop): ").strip()
    if raw == "":
        break
    try:
        age = int(raw)
        hp = int(input("Engine power in hp: "))
        km = int(input("Mileage in km: "))
        if min(age, hp, km) < 0 or hp == 0:
            raise ValueError
    except ValueError:
        print("Please enter valid positive whole numbers.\n")
        continue
    print(f"Estimated price: {predict_price(age, hp, km):,.0f} EGP\n")